# Session 2 lab — Data storage systems

One month of New York City yellow taxi trips, written down four different ways.
Same rows, same columns, same numbers — four files.

You will ask each file the same three questions and record two things: **how many
bytes the file forces you to read**, and how long it took. Then you will do the
same in PostgreSQL, with and without an index.

**You do not need to write any SQL.** Every query is already here. Run the cells
in order, top to bottom.

## Your results

### Part A — four representations

Times differ between laptops; the comparison between rows is the point.

| Format  | Size on disk | Q1 whole table | Q2 avg fare by hour | Q3 one trip |
|---------|--------------|----------------|---------------------|-------------|
| CSV     |              |                |                     |             |
| JSON    |              |                |                     |             |
| Parquet |              |                |                     |             |
| SQLite  |              |                |                     |             |

### Part B — PostgreSQL

|                       | Without index | With index |
|-----------------------|---------------|------------|
| One trip lookup       |               |            |
| Regulator's aggregate |               |            |

### Your conclusion

Three sentences: for the workload described in class, which system would you
choose, and what does it cost you?

## Part A — Four representations (12 min)

Every question below goes through the same engine, DuckDB, so what changes
between the rows of each table is the file, not the tool.

Alongside each time you get **must read** — the bytes that format gives the
engine no way to avoid touching. That number is a property of the file, not of
your laptop, and it is the one to pay attention to.

In [ ]:
import time
from pathlib import Path

import duckdb
import pyarrow.parquet as pq


def find_data():
    candidates = (
        Path("../../data/nyc-taxi"),  # from this notebook's own folder
        Path("data/nyc-taxi"),  # from the repository root
        Path("/app/data/nyc-taxi"),  # from anywhere inside the container
    )
    for candidate in candidates:
        if (candidate / "trips.parquet").exists():
            return candidate.resolve()
    raise SystemExit("No data. Run scripts/download_nyc_taxi.py first — see README.md")


DATA = find_data()
duckdb.sql("LOAD sqlite")  # lets DuckDB read a SQLite file too

FILES = {
    "CSV": DATA / "trips.csv",
    "JSON": DATA / "trips.jsonl",
    "Parquet": DATA / "trips.parquet",
    "SQLite": DATA / "trips.sqlite",
}

# The same table, named four ways, for the queries below.
SOURCES = {
    "CSV": f"read_csv('{FILES['CSV']}')",
    "JSON": f"read_json_auto('{FILES['JSON']}')",
    "Parquet": f"'{FILES['Parquet']}'",
    "SQLite": f"sqlite_scan('{FILES['SQLite']}', 'trips')",
}

for name, source in SOURCES.items():
    rows = duckdb.sql(f"SELECT count(*) FROM {source}").fetchone()[0]
    size = FILES[name].stat().st_size / 1_000_000
    print(f"{name:<8} {size:7,.0f} MB on disk   {rows:,} rows")

Same rows in every one of them, and nowhere near the same size. CSV and JSON
spell every number out as text, and JSON repeats all twenty column names on
every single row. Parquet stores each column separately, typed
and compressed.

Next, the machinery for the three questions — how much of a file must be read,
and how long the query takes.

In [ ]:
PARQUET = pq.ParquetFile(FILES["Parquet"])
META = PARQUET.metadata
COLUMN = {
    META.row_group(0).column(i).path_in_schema: i for i in range(META.num_columns)
}
GROUPS = [META.row_group(i) for i in range(META.num_row_groups)]

TRIP_ID = duckdb.sql(f"SELECT max(trip_id) // 2 FROM {SOURCES['Parquet']}").fetchone()[0]


def group_of(trip_id):
    """Which chunk holds this trip. Its min/max let DuckDB skip the others."""
    first = 1
    for index, group in enumerate(GROUPS):
        if trip_id < first + group.num_rows:
            return index
        first += group.num_rows
    return len(GROUPS) - 1


def must_read(name, columns=None, one_chunk=False):
    """Bytes the format gives you no way to avoid reading.

    CSV, JSON and SQLite have one answer to every question: the whole file. Only
    Parquet can read part of itself, because it keeps each column in its own
    chunk and records the smallest and largest value in each.
    """
    if name != "Parquet":
        return FILES[name].stat().st_size

    groups = [GROUPS[group_of(TRIP_ID)]] if one_chunk else GROUPS
    wanted = [COLUMN[column] for column in columns] if columns else list(COLUMN.values())
    return sum(
        group.column(index).total_compressed_size for group in groups for index in wanted
    )


def ask(question, sql_for, bytes_for):
    """Put one question to all four formats. Best of three, after a warm-up."""
    print(f"{question}\n")
    for name, source in SOURCES.items():
        sql = sql_for(source)
        duckdb.execute(sql).fetchall()  # warm the page cache
        best = min(_seconds(sql) for _ in range(3))
        megabytes = bytes_for(name) / 1_000_000
        print(f"  {name:<8} must read {megabytes:7,.1f} MB   {best:7.3f} s")
    print()


def _seconds(sql):
    started = time.perf_counter()
    duckdb.execute(sql).fetchall()
    return time.perf_counter() - started

### Q1 — Give me the whole table

The question a script asks when it starts by loading "the data" without thinking
about it. Every format has to hand over all twenty columns and every row, so
nobody can read less than everything.

Do not be surprised if JSON finishes this one faster than CSV despite reading
four times as many bytes — DuckDB simply throws more threads at it. **The bytes
are a property of the file; the seconds are a property of the reader.** That is
why the bytes are the column to trust.

In [ ]:
ask(
    "Q1  the whole table",
    lambda source: f"CREATE OR REPLACE TABLE everything AS SELECT * FROM {source}",
    lambda name: must_read(name),
)

### Q2 — The regulator's question

> *What was the average fare, by hour of the day, across the whole month?*

Still every row — but only two of the twenty columns. Watch what happens to
**must read**.

In [ ]:
REGULATOR = """
SELECT date_part('hour', tpep_pickup_datetime::TIMESTAMP) AS hour,
       round(avg(fare_amount), 2) AS avg_fare,
       count(*) AS trips
FROM {source}
GROUP BY hour
ORDER BY hour
"""

ask(
    "Q2  average fare by hour of day",
    lambda source: REGULATOR.format(source=source),
    lambda name: must_read(name, columns=["tpep_pickup_datetime", "fare_amount"]),
)

duckdb.sql(REGULATOR.format(source=SOURCES["Parquet"])).df().head(24)

Three of the four read exactly what they read before: they store rows, so the two
columns you want are interleaved with the eighteen you do not, and skipping them
is not possible. Parquet reads a fraction of itself, because each column sits in
its own chunk.

SQLite gets noticeably quicker here all the same — it still walks every page, but
it can skip *decoding* the columns nobody asked for. Fewer bytes read, no; less
work done, yes.

This is the whole argument for columnar storage, and it is why analytical
questions on Parquet are cheap.

### Q3 — One trip

A single row, by `trip_id`. This is the question none of these files is built
for: with no index, finding one row means checking every row.

Parquet gets away with less, and not because it is columnar. Each chunk records
the smallest and largest `trip_id` inside it, and we numbered the trips in order
when we built the file — so almost every chunk can be dismissed without being
read. Shuffle the rows and that advantage disappears.

In [ ]:
ask(
    f"Q3  the one trip with id {TRIP_ID:,}",
    lambda source: f"SELECT * FROM {source} WHERE trip_id = {TRIP_ID}",
    lambda name: must_read(name, one_chunk=True),
)

**Fill in the Part A table at the top now.**

Then answer this: which file would you hand to a colleague who has to answer the
regulator's question every morning, and which to one who has to open it in Excel?

Notice what is still missing. Three of these four files must be read end to end
to find one trip, and the fourth only escapes because we happened to number the
rows in order. None of them can *find* anything. That is what Part B is about.

## Part B — Reading every page, or the index (10 min)

Same data, now inside a database server: PostgreSQL, running in the same
`docker compose` that serves this notebook.

An index is a second structure beside the table that lets the server find rows
without looking at all of them. The interesting part is not that indexes make
things faster — it is *which* things.

In [ ]:
import io

import pandas as pd
import psycopg

POSTGRES = "host=postgres port=5432 user=labs password=labs dbname=labs"

COLUMNS = [
    "trip_id",
    "tpep_pickup_datetime",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount",
]

# Row-by-row INSERT of a million rows would take minutes. COPY is the bulk path.
trips = pd.read_parquet(FILES["Parquet"], columns=COLUMNS)
buffer = io.StringIO()
trips.to_csv(buffer, index=False, header=False)
buffer.seek(0)

connection = psycopg.connect(POSTGRES, autocommit=True)
cursor = connection.cursor()

started = time.perf_counter()
cursor.execute("DROP TABLE IF EXISTS trips")
cursor.execute("""
    CREATE TABLE trips (
        trip_id              integer,
        tpep_pickup_datetime timestamp,
        trip_distance        double precision,
        fare_amount          numeric,
        tip_amount           numeric,
        total_amount         numeric
    )
""")
with cursor.copy("COPY trips FROM STDIN WITH (FORMAT csv)") as copy:
    copy.write(buffer.read())
cursor.execute("ANALYZE trips")
print(f"{len(trips):,} rows copied in {time.perf_counter() - started:.1f} s")

### How to read a query plan

`EXPLAIN ANALYZE` shows how PostgreSQL decided to run a query. Two phrases are
worth recognising:

- **Seq Scan** — read every row in the table and check each one. (Sometimes
  *Parallel Seq Scan*: the same thing, split across CPU cores.)
- **Index Scan** — go to the index, jump straight to the rows that match.

In [ ]:
LOOKUP = f"SELECT * FROM trips WHERE trip_id = {TRIP_ID}"

REGULATOR_SQL = """
SELECT date_part('hour', tpep_pickup_datetime) AS hour,
       round(avg(fare_amount), 2) AS avg_fare,
       count(*) AS trips
FROM trips
GROUP BY hour
ORDER BY hour
"""


def explain(label, sql):
    """Time one query and report the scan PostgreSQL chose.

    Four runs: one to warm the cache, then three measured, keeping the fastest.
    A single measurement of something this quick is mostly noise.
    """
    cursor.execute(sql)
    cursor.fetchall()

    times, scan = [], "?"
    for _ in range(3):
        cursor.execute("EXPLAIN (ANALYZE, COSTS OFF) " + sql)
        plan = [row[0] for row in cursor.fetchall()]
        scan = next((line.strip() for line in plan if "Scan" in line), "?")
        times.append(
            next(
                float(line.split()[2])
                for line in plan
                if line.startswith("Execution Time:")
            )
        )

    best = min(times)
    print(f"{label:<26} {best:9.2f} ms   {scan}")
    return best

### B1 — Without an index

There is no index on `trip_id` yet. Watch what PostgreSQL has to do to find one
row — and note that it is the same thing Part A's files had to do.

In [ ]:
lookup_before = explain("lookup, no index", LOOKUP)
aggregate_before = explain("aggregate, no index", REGULATOR_SQL)

### B2 — Add the index, ask again

One line. It costs disk space, and time on every write, and in exchange the
server gets a way to find rows by `trip_id` without reading the table.

In [ ]:
cursor.execute("CREATE INDEX ON trips (trip_id)")

lookup_after = explain("lookup, with index", LOOKUP)
aggregate_after = explain("aggregate, with index", REGULATOR_SQL)


def verdict(before, after):
    """Anything under 2x here is noise, not a result."""
    ratio = before / after
    return f"{ratio:>6.0f}x faster" if ratio >= 2 else "     unchanged"


print()
print("=" * 62)
print(
    f"  one trip lookup       {lookup_before:8.2f} ->{lookup_after:8.2f} ms"
    f"   {verdict(lookup_before, lookup_after)}"
)
print(
    f"  regulator's aggregate {aggregate_before:8.2f} ->{aggregate_after:8.2f} ms"
    f"   {verdict(aggregate_before, aggregate_after)}"
)
print("=" * 62)

The index answers *where is one particular trip* without touching the table. It
does nothing at all for the average fare per hour, because that question needs
every row anyway — so PostgreSQL ignores the index and scans the table, exactly
as it did before.

An index is not a speed setting. It is an answer to one specific question.

**Fill in the Part B table.** Then: if someone told you to "add an index to make
the dashboard faster", what would you need to know first?

## Part C — The lost payout (8 min)

Not in this notebook: you need to watch two things happen at once. Open
**`PART_C.md`** and follow it with two terminals side by side.

## Stretch — only if you have finished everything else

Redis is running too. It is a key–value store: you put a value in under a key,
and you get it back by naming that key. That is the entire model.

In [ ]:
import redis

store = redis.Redis(host="redis", port=6379, decode_responses=True)

LOCATIONS = {
    4471: "40.7580,-73.9855",
    4472: "40.7128,-74.0060",
    4473: "40.7484,-73.9857",
    4474: "40.6413,-73.7781",
}
for driver_id, location in LOCATIONS.items():
    store.set(f"driver:{driver_id}:location", location)

print("driver 4471 is at", store.get("driver:4471:location"))
print("keys stored:", sorted(store.keys("driver:*")))

That lookup took microseconds, and would still take microseconds with a million
drivers. Now ask Redis the regulator's question.

There is no cell for it, because there is no query to write. Work out what you
*would* have to do, and write down two sentences on why this store cannot answer
the question. Two separate things are missing, not one.